# **Notebook 1: Data Cleaning & Order-Level Dataset Construction**
## Capstone: Customer Satisfaction Analysis for the Olist Brazilian E-Commerce Marketplace
---

### 📋 TO-DO: Before Running This Notebook

**What you NEED for this notebook:**
- [ ] A Kaggle account + API token (`kaggle.json`) so `kagglehub` can download the dataset
- [ ] Internet access (to download `olistbr/brazilian-ecommerce` from Kaggle)

**What this notebook will CREATE:**
- [ ] `olist_clean.csv` — a cleaned, **order-level** dataset (~96k rows) with the binary target `satisfied`.
  _(Required by: Notebook 2 — EDA & Baseline Model)_

> **Stage 1 scope (very important):** This notebook does **structural** cleaning only — merging the nine
> tables, restricting to delivered+reviewed orders, fixing types, and creating the target.
> We do **NOT** scale, impute, encode, or build statistical features (e.g., `seller_avg_score`) here.
> Those happen **after** the train/test split in Notebook 2 to prevent data leakage.
---

## Setup — Install & Import

**Hints & Tips:**
* `kagglehub` downloads the dataset to a local cache and returns the folder path.
* If you prefer, you can skip `kagglehub` and upload the 9 CSVs manually — just set `DATA_DIR` to that folder.

In [3]:
# Colab already ships pandas/numpy/matplotlib/seaborn — we only need kagglehub here.
# (Avoid `-U` on numpy/pandas in Colab: upgrading them mid-session breaks the kernel.)
!pip install -q kagglehub

In [4]:
import pandas as pd

In [5]:
import os, glob
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

RANDOM_STATE = 42  # used consistently across all notebooks for reproducibility

## **Stage 1: Data Understanding & Readiness**
### **Task 1.3: Load, Merge, and Inspect the Dataset**

#### **1.3.1 Load and Inspect Dataset Structure [2 marks]**
**The Task:** Download the Olist dataset from Kaggle and load all **nine** relational CSV tables. Print the
shape and schema of each so you understand what you are working with.

**Hints & Tips:**
* The dataset slug is `olistbr/brazilian-ecommerce` (~100k orders across 9 tables).
* Key tables: `orders`, `order_items`, `order_reviews`, `order_payments`, `products`, `sellers`,
  `customers`, `geolocation`, `product_category_name_translation`.
* Note the join keys: `order_id`, `customer_id`, `product_id`, `seller_id`.
* Watch for the misspelled product columns: `product_name_lenght`, `product_description_lenght`.

**Why we are doing it:** You cannot merge tables correctly until you know each table's grain and keys.

**🧠 Learner Inference:** Each table answers a different question about an order. Your job in 1.3.2 is to
fold them all onto a single row per `order_id`.

In [6]:
# Load the datasets [1 mark]
# YOUR CODE HERE
import kagglehub

# Download the Olist Brazilian E-Commerce dataset from Kaggle
dataset_path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Dataset downloaded to:")
print(dataset_path)

# Find all CSV files in the downloaded folder
csv_files = sorted(glob.glob(os.path.join(dataset_path, "*.csv")))

print(f"\nNumber of CSV files found: {len(csv_files)}")
print("\nCSV files:")
for file in csv_files:
    print(os.path.basename(file))

# Load all nine CSV files into pandas DataFrames

dataframes = {}

for file in csv_files:
    file_name = os.path.basename(file)
    table_name = file_name.replace(".csv", "")
    dataframes[table_name] = pd.read_csv(file)

print(f"Successfully loaded {len(dataframes)} datasets.")

# Inspect the datasets [1 mark]
# YOUR CODE HERE

for name, df in dataframes.items():
    print("=" * 80)
    print(f"Dataset: {name}")
    print(f"Shape: {df.shape}")
    print("\nColumns:")
    print(df.columns.tolist())
    print("\nData types:")
    print(df.dtypes)
    print()

Dataset downloaded to:
C:\Users\Niroj\.cache\kagglehub\datasets\olistbr\brazilian-ecommerce\versions\2

Number of CSV files found: 9

CSV files:
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv
Successfully loaded 9 datasets.
Dataset: olist_customers_dataset
Shape: (99441, 5)

Columns:
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Data types:
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

Dataset: olist_geolocation_dataset
Shape: (1000163, 5)

Columns:
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

Data types:
geolo

#### **1.3.2 Merge to an Order-Level Grain [3 marks]**
**The Task:** Collapse the nine tables into **one row per `order_id`**. Several tables have a finer grain
than the order (an order has many items and may have several payment records), so they must be
**aggregated before joining** — otherwise the merge multiplies rows and corrupts the target distribution.

**Hints & Tips:**
* **order_items → order level:** count items, **sum** `price` and `freight_value`. Pick a *primary item*
  (the most expensive line) to carry the representative `product_id` and `seller_id`.
* **order_payments → order level:** **sum** `payment_value`, take **max** `payment_installments`, count
  payment records, and take the **dominant** `payment_type` (highest-value record).
* **order_reviews → order level:** a few orders have multiple reviews — keep the **latest** one per order.
* Build the spine as `orders → customers`, then left-join the aggregated items/payments/reviews, then the
  primary product (+ English category) and seller.

**🔧 Design choice to document:** using the *most expensive* line as the primary item is a defensible rule
for multi-item orders.

**🧠 Learner Inference:** If your merged row count is wildly above ~100k, you forgot to aggregate a
one-to-many table before joining.

In [7]:
# ---- Aggregate order_items to order level ---- [1 mark]
# YOUR CODE HERE
# Get the order_items dataset
order_items = dataframes["olist_order_items_dataset"]
# Aggregate item-level information to one row per order
order_items_agg = (
    order_items
    .groupby("order_id", as_index=False)
    .agg(
        n_items=("order_item_id", "count"),
        price=("price", "sum"),
        freight_value=("freight_value", "sum")
    )
)

# Inspect the result
print("Shape:", order_items_agg.shape)
order_items_agg.head()

Shape: (98666, 4)


,order_id,n_items,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14


In [8]:
# Primary item = most expensive line in the order (carries representative product & seller)
# YOUR CODE HERE

# Sort items within each order by price descending.
# order_item_id is used as a deterministic tie-breaker.
items_sorted = order_items.sort_values(
    ["order_id", "price", "order_item_id"],
    ascending=[True, False, True]
)

# Select the most expensive item from each order
primary_items = (
    items_sorted
    .drop_duplicates(subset="order_id", keep="first")
    [["order_id", "product_id", "seller_id"]]
)

print("Shape:", primary_items.shape)
primary_items.head()

Shape: (98666, 3)


,order_id,product_id,seller_id
0,00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202
1,00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36
2,000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d
3,00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4
4,00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87


In [9]:

# ---- Aggregate order_payments to order level ----
# YOUR CODE HERE

# Get the payments dataset
order_payments = dataframes["olist_order_payments_dataset"]

# Aggregate payment information to one row per order
payments_agg = (
    order_payments
    .groupby("order_id", as_index=False)
    .agg(
        payment_value=("payment_value", "sum"),
        payment_installments=("payment_installments", "max"),
        n_payments=("payment_sequential", "count")
    )
)

print("Shape:", payments_agg.shape)
payments_agg.head()

Shape: (99440, 4)


,order_id,payment_value,payment_installments,n_payments
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2,1
1,00018f77f2f0320c557190d7a144bdd3,259.83,3,1
2,000229ec398224ef6ca0657da4fc703e,216.87,5,1
3,00024acbcdf0a6daa1e931b038114c75,25.78,2,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3,1


In [10]:
# Dominant payment type = type of the highest-value payment record
# YOUR CODE HERE

# Sort payment records by order and payment value.
# payment_sequential provides a deterministic tie-breaker.
payments_sorted = order_payments.sort_values(
    ["order_id", "payment_value", "payment_sequential"],
    ascending=[True, False, True]
)

# Select the highest-value payment record for each order
dominant_payment = (
    payments_sorted
    .drop_duplicates(subset="order_id", keep="first")
    [["order_id", "payment_type"]]
    .rename(columns={"payment_type": "dominant_payment_type"})
)

print("Shape:", dominant_payment.shape)
dominant_payment.head()

Shape: (99440, 2)


,order_id,dominant_payment_type
85283,00010242fe8c5a6d1ba2dd792cb16214,credit_card
2499,00018f77f2f0320c557190d7a144bdd3,credit_card
12393,000229ec398224ef6ca0657da4fc703e,credit_card
32971,00024acbcdf0a6daa1e931b038114c75,credit_card
98711,00042b26cf59d7ce69dfabb4e55b4fd9,credit_card


In [11]:
# ---- Reduce reviews to one (latest) per order ---- [1 mark]
# YOUR CODE HERE

reviews = dataframes["olist_order_reviews_dataset"].copy()

# Convert review creation date to datetime for correct chronological sorting
reviews["review_creation_date"] = pd.to_datetime(
    reviews["review_creation_date"],
    errors="coerce"
)

# Sort by order and latest review date
reviews_sorted = reviews.sort_values(
    ["order_id", "review_creation_date", "review_id"],
    ascending=[True, False, False]
)

# Keep the latest review for each order
reviews_latest = (
    reviews_sorted
    .drop_duplicates(subset="order_id", keep="first")
)

print("Shape:", reviews_latest.shape)
print("Unique orders:", reviews_latest["order_id"].nunique())
print("Duplicate order IDs:", reviews_latest["order_id"].duplicated().sum())



Shape: (98673, 7)
Unique orders: 98673
Duplicate order IDs: 0


In [12]:
# ---- Build the order-level spine and join everything on order_id ----
# YOUR CODE HERE

orders = dataframes["olist_orders_dataset"].copy()
customers = dataframes["olist_customers_dataset"].copy()

order_level = orders.copy()

# Join customer information
order_level = order_level.merge(
    customers,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

# Join aggregated order-item information
order_level = order_level.merge(
    order_items_agg,
    on="order_id",
    how="left",
    validate="one_to_one"
)

# Join aggregated payment information
order_level = order_level.merge(
    payments_agg,
    on="order_id",
    how="left",
    validate="one_to_one"
)

# Attach dominant payment type
order_level = order_level.merge(
    dominant_payment,
    on="order_id",
    how="left",
    validate="one_to_one"
)
   

# Join the latest review for each order
order_level = order_level.merge(
    reviews_latest,
    on="order_id",
    how="left",
    validate="one_to_one"
)

# Check that the order-level grain has been preserved
print("Shape after merging orders, customers, items, payments and reviews:",
      order_level.shape)
print("Unique order IDs:", order_level["order_id"].nunique())
print("Duplicate order IDs:",
      order_level["order_id"].duplicated().sum())

Shape after merging orders, customers, items, payments and reviews: (99441, 25)
Unique order IDs: 99441
Duplicate order IDs: 0


In [13]:
# ---- Attach product attributes (+ English category) via the primary product_id ---- [1 mark]
# YOUR CODE HERE
products = dataframes["olist_products_dataset"].copy()
category_translation = dataframes["product_category_name_translation"].copy()

# Attach primary product information to the order-level dataset
order_level = order_level.merge(
    primary_items,
    on="order_id",
    how="left",
    validate="one_to_one"
)

# Attach product attributes using product_id
order_level = order_level.merge(
    products,
    on="product_id",
    how="left",
    validate="many_to_one"
)

# Attach English product category
order_level = order_level.merge(
    category_translation,
    on="product_category_name",
    how="left",
    validate="many_to_one"
)

print("Shape after attaching product information:", order_level.shape)
print("Unique order IDs:", order_level["order_id"].nunique())
print("Duplicate order IDs:",
      order_level["order_id"].duplicated().sum())


Shape after attaching product information: (99441, 36)
Unique order IDs: 99441
Duplicate order IDs: 0


In [14]:
# ---- Attach seller state via the primary seller_id ----
# YOUR CODE HERE
sellers = dataframes["olist_sellers_dataset"].copy()

# Keep only the fields required for the order-level dataset
seller_info = sellers[["seller_id", "seller_state"]].copy()

# Attach seller state
order_level = order_level.merge(
    seller_info,
    on="seller_id",
    how="left",
    validate="many_to_one"
)

print("seller_state" in order_level.columns)
print("Shape:", order_level.shape)
print("Duplicate order IDs:",
      order_level["order_id"].duplicated().sum())


True
Shape: (99441, 37)
Duplicate order IDs: 0


#### **1.3.3 Profile Missing Values, Duplicates, and Ranges [1 mark]**
**The Task:** Profile the merged table: missing values per column, duplicate `order_id`s, and the numeric
ranges. This tells you what cleaning is required in Task 1.4.

**Hints & Tips:**
* `df.isnull().mean().sort_values(ascending=False)` gives the missing-fraction per column.
* Undelivered / cancelled orders are *expected* to have missing delivery timestamps — that's why we filter
  to delivered orders next.
* `df.describe()` reveals impossible values (e.g., negative freight) if any exist.

**🧠 Learner Inference:** High missingness in `order_delivered_customer_date` and `review_score` simply
reflects orders that were never delivered or never reviewed — they will be removed in 1.4.1.

In [15]:
# Data Profiling [1 mark]
# YOUR CODE HERE

# Missing values per column
missing_fraction = (
    order_level.isnull()
    .mean()
    .sort_values(ascending=False)
)

print("Missing value fraction by column:")
print(missing_fraction)


# Duplicate order IDs
duplicate_order_ids = order_level["order_id"].duplicated().sum()

print("\nDuplicate order IDs:", duplicate_order_ids)


# Numeric column ranges
print("\nNumeric column summary:")
print(order_level.describe().T)

Missing value fraction by column:
review_comment_title             0.883831
review_comment_message           0.589908
order_delivered_customer_date    0.029817
product_category_name_english    0.022244
product_category_name            0.022023
product_photos_qty               0.022023
product_description_lenght       0.022023
product_name_lenght              0.022023
order_delivered_carrier_date     0.017930
product_width_cm                 0.007954
product_height_cm                0.007954
product_length_cm                0.007954
product_weight_g                 0.007954
seller_state                     0.007794
product_id                       0.007794
seller_id                        0.007794
n_items                          0.007794
freight_value                    0.007794
price                            0.007794
review_id                        0.007723
review_score                     0.007723
review_answer_timestamp          0.007723
review_creation_date             0.007723


### **Task 1.4: Clean and Prepare Analysis-Ready Data**

#### **1.4.1 Restrict to Delivered+Reviewed Orders & Remove Structural Issues [2 marks]**
**The Task:** Keep only orders that (a) were actually **delivered** and (b) have a **review score** — both
are required to build and validate the target. Then drop structural duplicates and rows missing the
essential fields.

**Hints & Tips:**
* Filter `order_status == "delivered"`.
* Require non-null `review_score` **and** non-null `order_delivered_customer_date`
  (a handful of "delivered" rows lack the actual delivery timestamp — drop those).
* Drop duplicate `order_id`s if any survived.
* Do **not** drop rows just because a *product attribute* is missing — leave those NaNs for Notebook 2 to
  handle (imputation is a post-split step).

**🧠 Learner Inference:** Restricting to delivered+reviewed is what defines our modelling population: orders
where a customer actually experienced fulfilment and told us how they felt.

In [16]:
# (a) delivered orders only [1 mark]
# YOUR CODE HERE

order_level = order_level[
    order_level["order_status"] == "delivered"
].copy()

print("Shape after filtering delivered orders:", order_level.shape)
print("\nOrder status counts:")
print(order_level["order_status"].value_counts())


Shape after filtering delivered orders: (96478, 37)

Order status counts:
order_status
delivered    96478
Name: count, dtype: int64


In [17]:
# (b) must have a review score
# YOUR CODE HERE

order_level = order_level[
    order_level["review_score"].notna()
    & order_level["order_delivered_customer_date"].notna()
].copy()

print("Shape after requiring review score and delivery date:",
      order_level.shape)

print("Missing review scores:",
      order_level["review_score"].isna().sum())

print("Missing delivery dates:",
      order_level["order_delivered_customer_date"].isna().sum())

Shape after requiring review score and delivery date: (95824, 37)
Missing review scores: 0
Missing delivery dates: 0


In [18]:
# (c) must have an actual delivery date [1 mark]
# YOUR CODE HERE
order_level = order_level[
    order_level["order_delivered_customer_date"].notna()
].copy()

print("Shape after requiring actual delivery date:", order_level.shape)
print(
    "Missing actual delivery dates:",
    order_level["order_delivered_customer_date"].isna().sum()
)


Shape after requiring actual delivery date: (95824, 37)
Missing actual delivery dates: 0


In [19]:
# (d) drop structural duplicates
# YOUR CODE HERE

order_level = order_level.drop_duplicates(
    subset="order_id",
    keep="first"
).copy()

print("Final shape after structural cleaning:", order_level.shape)
print("Unique order IDs:", order_level["order_id"].nunique())
print(
    "Duplicate order IDs:",
    order_level["order_id"].duplicated().sum()
)

Final shape after structural cleaning: (95824, 37)
Unique order IDs: 95824
Duplicate order IDs: 0


#### **1.4.2 Standardise Schema, Categories, and Timestamps [3 marks]**
**The Task:** Convert all date columns to real datetimes, standardise text fields (lowercase/strip the
English category and payment type), and tidy dtypes so Notebook 2 can engineer features cleanly.

**Hints & Tips:**
* Parse every `*_date` / `*_timestamp` column with `pd.to_datetime(..., errors="coerce")`.
* Fill the English `product_category` for untranslated categories by falling back to the original
  Portuguese name (rather than leaving it blank).
* Strip/lowercase categorical strings so `"Credit_Card "` and `"credit_card"` don't become two categories.

**🧠 Learner Inference:** Standardising now means the delivery-delay and processing-time features you build
in Notebook 2 are just clean date subtractions.

In [20]:
# ---- Parse all date/time columns ---- [1 mark]
# YOUR CODE HERE
date_cols = [
    col for col in order_level.columns
    if col.endswith("_date") or col.endswith("_timestamp")
]

for col in date_cols:
    order_level[col] = pd.to_datetime(
        order_level[col],
        errors="coerce"
    )

print("Date/time columns converted:")
print(date_cols)

print("\nDtypes after conversion:")
print(order_level[date_cols].dtypes)


Date/time columns converted:
['order_purchase_timestamp', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'review_creation_date', 'review_answer_timestamp']

Dtypes after conversion:
order_purchase_timestamp         datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
review_creation_date             datetime64[us]
review_answer_timestamp          datetime64[us]
dtype: object


In [21]:
# ---- Standardise categorical text ---- [1 mark]
# Fall back to Portuguese name where English translation is missing
# YOUR CODE HERE

order_level["product_category_name_english"] = (
    order_level["product_category_name_english"]
    .fillna(order_level["product_category_name"])
)

# Standardise product category text
order_level["product_category_name_english"] = (
    order_level["product_category_name_english"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Standardise payment type
order_level["dominant_payment_type"] = (
    order_level["dominant_payment_type"]
    .astype("string")
    .str.strip()
    .str.lower()
)

print("Sample product categories:")
print(order_level["product_category_name_english"].head())

print("Payment types:")
print(order_level["dominant_payment_type"].value_counts(dropna=False))

Sample product categories:
0    housewares
1     perfumery
2          auto
3      pet_shop
4    stationery
Name: product_category_name_english, dtype: string
Payment types:
dominant_payment_type
credit_card    72331
boleto         19062
voucher         2953
debit_card      1477
<NA>               1
Name: count, dtype: Int64


In [22]:
# ---- Tidy a couple of dtypes ---- [1 mark]
# YOUR CODE HERE

# Convert count/score columns to nullable integer type
order_level["n_items"] = order_level["n_items"].astype("Int64")
order_level["n_payments"] = order_level["n_payments"].astype("Int64")
order_level["review_score"] = order_level["review_score"].astype("Int64")

print(order_level[[
    "n_items",
    "n_payments",
    "review_score"
]].dtypes)

n_items         Int64
n_payments      Int64
review_score    Int64
dtype: object


#### **1.4.3 Create the Binary Target & Export the Leakage-Safe Dataset [2 marks]**
**The Task:** Create the binary classification target and export the analysis-ready, **split-ready** file
for Notebook 2.

**Hints & Tips:**
* `satisfied = (review_score >= 4).astype(int)` → 1 = satisfied (4–5★), 0 = unsatisfied (1–3★).
* Expect roughly **79% satisfied / 21% unsatisfied** — confirm this and note it.
* Keep the raw delivery timestamps and raw numeric/categorical columns; Notebook 2 will derive
  `delivery_delay_days`, `freight_ratio`, etc., **after** the split.

**⛔ Leakage guardrail:** Do **NOT** apply `StandardScaler`, mean/median imputation, target/frequency
encoding, or `seller_avg_score` here. Any statistic learned from the whole dataset would leak test
information into training. Those steps belong in Notebook 2, fit on the training split only.

**🧠 Learner Inference:** A clean, untransformed file is the contract with Notebook 2: it hands over *facts*
about each order, not *statistics* derived from the full population.

In [23]:
# ---- Binary target ---- [1 mark]
# YOUR CODE HERE

# Create binary satisfaction target
order_level["satisfied"] = (
    order_level["review_score"] >= 4
).astype(int)

# Check target distribution
print(order_level["satisfied"].value_counts())
print(
    (order_level["satisfied"].value_counts(normalize=True) * 100).round(2)
)

satisfied
1    75634
0    20190
Name: count, dtype: int64
satisfied
1    78.93
0    21.07
Name: proportion, dtype: float64


In [24]:
# ---- Select the columns we hand to Notebook 2 (raw facts, no derived statistics) ---- [1 mark]
# YOUR CODE HERE

final_columns = [
    "order_id",
    "customer_id",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state",
    
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    
    "n_items",
    "price",
    "freight_value",
    
    "payment_value",
    "payment_installments",
    "n_payments",
    "dominant_payment_type",
    
    "review_score",
    
    "product_id",
    "product_category_name",
    "product_category_name_english",
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
    
    "seller_id",
    "seller_state",
    
    "satisfied"
]

olist_clean = order_level[final_columns].copy()

print("Final dataset shape:", olist_clean.shape)
print("\nFinal columns:")
print(olist_clean.columns.tolist())

Final dataset shape: (95824, 33)

Final columns:
['order_id', 'customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'n_items', 'price', 'freight_value', 'payment_value', 'payment_installments', 'n_payments', 'dominant_payment_type', 'review_score', 'product_id', 'product_category_name', 'product_category_name_english', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'seller_id', 'seller_state', 'satisfied']


---
## 💾 Save Artifact for Notebook 2

**IMPORTANT:** This cell writes `olist_clean.csv`. Notebook 2 depends on this file.

In [25]:
# Save the cleaned dataset
# YOUR CODE HERE

olist_clean.to_csv("olist_clean.csv", index=False)

print("olist_clean.csv saved successfully.")
print("Shape:", olist_clean.shape)

olist_clean.csv saved successfully.
Shape: (95824, 33)


In [26]:
print("File exists:", os.path.exists("olist_clean.csv"))

File exists: True


---
## ✅ END-OF-NOTEBOOK CHECKLIST

> **⚠️ Verify every item before moving to Notebook 2.**

- [ ] All **nine** Olist tables loaded and inspected (`.info()` / shapes printed)
- [ ] Tables merged to **one row per `order_id`** with documented grain collapse (items, payments, reviews)
- [ ] Missing values, duplicates, and numeric ranges profiled
- [ ] Restricted to **delivered + reviewed** orders; structural duplicates / absolute-missing rows removed
- [ ] Timestamps parsed; product categories translated to English; categorical text standardised
- [ ] Binary target `satisfied` created and class balance (~79/21) confirmed
- [ ] **NO** scaling, imputation, encoding, or `seller_avg_score` applied (leakage prevention)
- [ ] **`olist_clean.csv` saved to disk** ← _CRITICAL for Notebook 2_

**If any item is unchecked, fix it before proceeding.**